<a href="https://colab.research.google.com/github/yashikasgh/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashikasgh/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [4]:

# ML-05: Build the feature vector

import pandas as pd
import numpy as np

# Reuse df if the dataset is already loaded.
# Otherwise, upload the anonymized CSV.
if "df" not in globals():
    from google.colab import files
    uploaded = files.upload()
    csv_files = [name for name in uploaded if name.endswith(".csv")]

    if not csv_files:
        raise ValueError("Please upload the anonymized dataset CSV.")

    df = pd.read_csv(csv_files[0])

print("Dataset shape:", df.shape)
print("Unique content IDs:", df["content_id"].nunique()
      if "content_id" in df.columns else "Column unavailable")

# Define the target separately; do not use it as a feature.
if "trend_direction" not in df.columns:
    raise ValueError("Expected column 'trend_direction' is missing.")

y = (df["trend_direction"] == "down").astype(int)

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
]

features = numeric_features + categorical_features
missing_columns = [col for col in features if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing feature columns: {missing_columns}")

X = df[features].copy()

# Numeric missing values: median imputation.
for col in numeric_features:
    X[col] = pd.to_numeric(X[col], errors="coerce")
    X[col] = X[col].fillna(X[col].median())

# Categorical missing values: explicit category.
for col in categorical_features:
    X[col] = X[col].astype("string").fillna("Unknown")

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Down-label prevalence:", round(y.mean(), 3))
print("Remaining missing feature values:", int(X.isna().sum().sum()))
print("Feature columns:", features)


Dataset shape: (30000, 44)
Unique content IDs: 30000
Feature matrix shape: (30000, 14)
Target shape: (30000,)
Down-label prevalence: 0.542
Remaining missing feature values: 0
Feature columns: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier']


## 2. Feature notes (meaning, missing, categorical, available-when?)


### Feature notes

| Feature | Meaning | Missing-value handling | Available before prediction? |
|---|---|---|---|
| `search_volume` | Search demand estimate | Numeric median imputation | To be verified against the source snapshot |
| `competition` | Competition measure | Numeric median imputation | To be verified against the source snapshot |
| `cpc` | Cost-per-click estimate | Numeric median imputation | To be verified against the source snapshot |
| `word_count` | Content word count | Numeric median imputation | Likely, if computed from the content version available at prediction time |
| `char_count` | Content character count | Numeric median imputation | Likely, if computed from the content version available at prediction time |
| `content_age_days` | Estimated content age in days | Numeric median imputation | To be verified against the source snapshot |
| `days_since_last_update` | Days since the last recorded update | Numeric median imputation | To be verified against the source snapshot |
| `competition_level` | Categorical competition band | Missing values set to `Unknown` | To be verified against the source snapshot |
| `content_type` | Content format/type | Missing values set to `Unknown` | To be verified against the source snapshot |
| `main_intent` | Main search intent category | Missing values set to `Unknown` | To be verified against the source snapshot |
| `age_tier` | Content-age category | Missing values set to `Unknown` | To be verified against the source snapshot |
| `freshness_tier` | Content-freshness category | Missing values set to `Unknown` | To be verified against the source snapshot |
| `word_count_tier` | Word-count category | Missing values set to `Unknown` | To be verified against the source snapshot |
| `char_count_tier` | Character-count category | Missing values set to `Unknown` | To be verified against the source snapshot |

### Timing and leakage caveat

These features are candidates, not proof of leakage-free prediction. Their availability must be checked against the actual prediction timestamp and source data definitions. Median imputation should be fitted on training data only when evaluating a model, then applied unchanged to validation/test data. The target `trend_direction` is used only to construct the label and is not included in the feature matrix.


In [5]:

# ML-05: Feature notes and data-quality checks

# Confirm the feature matrix from Section 1 exists.
if "X" not in globals() or "y" not in globals():
    raise RuntimeError("Run Section 1 first to create X and y.")

feature_notes = pd.DataFrame({
    "feature": X.columns,
    "dtype": [str(X[c].dtype) for c in X.columns],
    "missing_values": [int(X[c].isna().sum()) for c in X.columns],
    "unique_values": [int(X[c].nunique(dropna=False)) for c in X.columns],
})

print("FEATURE SUMMARY")
display(feature_notes)

print("\nCATEGORICAL VALUE EXAMPLES")
for col in categorical_features:
    print(f"{col}: {X[col].dropna().unique()[:8].tolist()}")

print("\nTARGET SUMMARY")
print(y.value_counts().rename(index={0: "not_down", 1: "down"}))
print("Target prevalence:", round(float(y.mean()), 3))

# Check whether the target or obvious target-derived fields entered X.
forbidden_exact = {
    "trend_direction",
    "trend_pct",
    "is_down",
    "baseline_score",
    "content_id",
}

leaked_exact = sorted(set(X.columns) & forbidden_exact)

print("\nFORBIDDEN-FIELD CHECK")
print("Forbidden fields found in X:", leaked_exact)
assert not leaked_exact, "Remove forbidden target/identifier fields from X."

print("\nMissing values remaining in X:", int(X.isna().sum().sum()))
print("Feature notes checks completed.")


FEATURE SUMMARY


,feature,dtype,missing_values,unique_values
0,search_volume,float64,0,41
1,competition,float64,0,101
2,cpc,float64,0,915
3,word_count,float64,0,5476
4,char_count,float64,0,14839
5,content_age_days,int64,0,225
6,days_since_last_update,int64,0,57
7,competition_level,string,0,4
8,content_type,string,0,3
9,main_intent,string,0,5



CATEGORICAL VALUE EXAMPLES
competition_level: ['HIGH', 'LOW', 'MEDIUM', 'Unknown']
content_type: ['keyword article', 'feedly article', 'comparison article']
main_intent: ['transactional', 'informational', 'commercial', 'Unknown', 'navigational']
age_tier: ['181-365', '365+', '91-180', '31-90']
freshness_tier: ['0-30', '91-180', '181+', '31-90']
word_count_tier: ['2000-3500', '3500+', 'Unknown', '<1000', '1000-2000']
char_count_tier: ['15000-25000', 'Unknown', '<8000', '8000-15000', '25000+']

TARGET SUMMARY
trend_direction
down        16262
not_down    13738
Name: count, dtype: int64
Target prevalence: 0.542

FORBIDDEN-FIELD CHECK
Forbidden fields found in X: []

Missing values remaining in X: 0
Feature notes checks completed.


## 3. The leakage hunt

The prediction target is defined as `is_down = 1` when `trend_direction == "down"`, and `0` otherwise.

The following fields are excluded from the feature matrix:

- `trend_direction`: directly defines the target.
- `trend_pct`: may encode information closely related to the target trend.
- Recent and previous 30-day performance metrics: excluded because their measurement windows and overlap with the target period have not been independently verified.
- `content_id` and other identifiers: identifiers are not intended as predictive signals.
- `baseline_score`: excluded to prevent the model from learning from the baseline used for comparison.

The current checks are a conservative column-level audit, not proof that the dataset is leakage-free. Exact label-generation timestamps and feature measurement windows are not fully established. In particular, the use of 90-day aggregates requires further temporal verification before the model can be described as a deployable forecasting system.


In [6]:

# ML-05: Leakage hunt

if "df" not in globals() or "X" not in globals():
    raise RuntimeError("Run Sections 1 and 2 first.")

columns = list(df.columns)

# Fields directly related to the target or identifiers.
target_related = [
    c for c in columns
    if c.lower() in {"trend_direction", "trend_pct", "is_down",
                     "baseline_score"}
]

identifier_fields = [
    c for c in columns
    if any(term in c.lower() for term in
           ["content_id", "client_id", "url", "domain"])
]

# Flag recent/previous-window metrics for manual temporal review.
window_terms = ["30d", "30_day", "30-day", "recent", "previous",
                "prior", "last_30", "prev_30"]

window_fields = [
    c for c in columns
    if any(term in c.lower() for term in window_terms)
]

# Report all columns flagged for review.
audit = pd.DataFrame({
    "column": columns,
    "in_feature_matrix": [c in X.columns for c in columns],
    "target_related": [c in target_related for c in columns],
    "identifier_candidate": [c in identifier_fields for c in columns],
    "time_window_review": [c in window_fields for c in columns],
})

print("TARGET-RELATED FIELDS")
print(target_related)

print("\nIDENTIFIER CANDIDATES")
print(identifier_fields)

print("\nTIME-WINDOW FIELDS REQUIRING REVIEW")
print(window_fields)

print("\nFLAGGED COLUMNS INCLUDED IN X")
flagged = set(target_related + identifier_fields + window_fields)
included = sorted(flagged.intersection(X.columns))
print(included)

# Strict checks for known forbidden fields.
forbidden = {"trend_direction", "trend_pct", "is_down",
             "baseline_score", "content_id"}
assert not (set(X.columns) & forbidden), (
    "Forbidden field(s) found in X: "
    + str(sorted(set(X.columns) & forbidden))
)

display(audit[audit[
    ["target_related", "identifier_candidate", "time_window_review"]
].any(axis=1)])

print("\nAudit complete. Flags identify review candidates; "
      "they do not independently prove temporal leakage.")


TARGET-RELATED FIELDS
['trend_direction', 'trend_pct']

IDENTIFIER CANDIDATES
['content_id', 'client_id']

TIME-WINDOW FIELDS REQUIRING REVIEW
['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']

FLAGGED COLUMNS INCLUDED IN X
[]


,column,in_feature_matrix,target_related,identifier_candidate,time_window_review
0,content_id,False,False,True,False
1,client_id,False,False,True,False
22,impressions_last_30d,False,False,False,True
23,clicks_last_30d,False,False,False,True
24,sessions_last_30d,False,False,False,True
25,impressions_prev_30d,False,False,False,True
26,clicks_prev_30d,False,False,False,True
27,sessions_prev_30d,False,False,False,True
42,trend_direction,False,True,False,False
43,trend_pct,False,True,False,False



Audit complete. Flags identify review candidates; they do not independently prove temporal leakage.


## 4. What I excluded and why


### Excluded fields and rationale

| Field or field group | Reason for exclusion |
|---|---|
| `trend_direction` | Directly used to construct the prediction target; including it would leak the answer. |
| `trend_pct` | Closely related to trend movement and potentially target-derived. |
| Recent/previous 30-day metrics | Excluded because their time windows may overlap with the target period. |
| `content_id` and other identifiers | Excluded to avoid identifier-based shortcuts and improve generalization. |
| Client/domain identifiers, if present | Not used as predictive features; grouped validation is used to assess performance on unseen clients. |
| `baseline_score` | Excluded so the model does not use the baseline score it is being compared against. |

### Remaining risks

Column exclusion alone cannot establish that all leakage has been removed. The exact target-generation window and feature timestamps still require verification. Some aggregate metrics may contain information from periods that overlap with the target. The results should therefore be interpreted as an exploratory, offline evaluation rather than proof of future performance in production.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.